# 🛰️ AeroSync: Unified Satellite Foundation Master Model (5-in-1)
### State-of-the-Art Deep Learning Techniques for Satellite & Remote Sensing Imagery
#### (Adapted from https://github.com/thatvivekhingu/techniques / satellite-image-deep-learning)

This single consolidated notebook contains all **5 Core Satellite Remote Sensing Deep Learning Models**:
1. **FarSeg (Foreground-Aware Relation Network)**: Wide-area semantic segmentation solving foreground-background imbalance.
2. **Oriented Bounding Box (OBB) Rotated Object Detector**: Arbitrary-angle rotated detection $(cx, cy, w, h, 	heta)$.
3. **Deep PanNet Residual Pansharpening**: MS + PAN 4x fusion with SAM & ERGAS quality metrics.
4. **CloudShadowNet Atmospheric Quality Control**: 4-class cloud, cirrus haze, and shadow segmentation.
5. **GeoFoundation Vision Transformer (GeoViT)**: Patch-based foundation embeddings & CBIR similarity search.


In [ ]:
# Environment Setup & Imports
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Union

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")


## 1. FarSeg (Foreground-Aware Relation Network for VHR Imagery)
Models global scene representation to suppress false alarms across vast agricultural & natural terrains.


In [ ]:
# --- 1. FarSeg Architecture ---
class ConvBNReLU(nn.Module):
    def __init__(self, in_c: int, out_c: int, k: int = 3, s: int = 1, p: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class FSRelationModule(nn.Module):
    def __init__(self, scene_channels: int, feature_channels: int, out_channels: int = 128):
        super().__init__()
        self.scene_pool = nn.AdaptiveAvgPool2d(1)
        self.scene_project = nn.Sequential(
            nn.Linear(scene_channels, out_channels),
            nn.ReLU(inplace=True),
            nn.Linear(out_channels, feature_channels),
            nn.Sigmoid(),
        )
        self.scale_conv = ConvBNReLU(feature_channels, out_channels, k=1, p=0)

    def forward(self, feat: torch.Tensor, scene_feat: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = scene_feat.shape
        vec = self.scene_pool(scene_feat).view(b, c)
        attn = self.scene_project(vec).unsqueeze(-1).unsqueeze(-1)
        return self.scale_conv(feat * attn + feat)

class FarSeg(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 5, fpn_dim: int = 64):
        super().__init__()
        # 4-stage encoder
        self.enc1 = ConvBNReLU(in_channels, 32, s=2)
        self.enc2 = ConvBNReLU(32, 64, s=2)
        self.enc3 = ConvBNReLU(64, 128, s=2)
        self.enc4 = ConvBNReLU(128, 256, s=2)

        self.lat4 = nn.Conv2d(256, fpn_dim, 1)
        self.lat3 = nn.Conv2d(128, fpn_dim, 1)
        self.lat2 = nn.Conv2d(64, fpn_dim, 1)

        self.fs_relation = FSRelationModule(scene_channels=256, feature_channels=fpn_dim, out_channels=fpn_dim)
        self.head = nn.Sequential(ConvBNReLU(fpn_dim, fpn_dim), nn.Conv2d(fpn_dim, num_classes, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sz = x.shape[-2:]
        c1 = self.enc1(x)
        c2 = self.enc2(c1)
        c3 = self.enc3(c2)
        c4 = self.enc4(c3)

        p4 = self.lat4(c4)
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="bilinear", align_corners=False)

        enhanced = self.fs_relation(p2, c4)
        logits = self.head(enhanced)
        return F.interpolate(logits, size=sz, mode="bilinear", align_corners=False)


## 2. Oriented Bounding Box (OBB) Rotated Object Detector
Detects tilted aerial structures, solar installations, vessels, and vehicles with rotation angle $	heta$.


In [ ]:
# --- 2. Oriented BBox Detector ---
@dataclass
class OrientedBox:
    cx: float
    cy: float
    width: float
    height: float
    angle_rad: float
    score: float = 1.0

    def to_corners(self) -> np.ndarray:
        cos_a, sin_a = math.cos(self.angle_rad), math.sin(self.angle_rad)
        w2, h2 = self.width / 2.0, self.height / 2.0
        corners = np.array([[-w2, -h2], [w2, -h2], [w2, h2], [-w2, h2]], dtype=np.float32)
        rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]], dtype=np.float32)
        rotated = np.dot(corners, rot.T)
        rotated[:, 0] += self.cx
        rotated[:, 1] += self.cy
        return rotated

class OrientedCadastralDetector(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 5, hidden_dim: int = 64):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, 7, stride=2, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, hidden_dim, 3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
        )
        self.cls_head = nn.Conv2d(hidden_dim, num_classes, 1)
        self.reg_head = nn.Conv2d(hidden_dim, 6, 1)  # (dx, dy, log_w, log_h, sin_th, cos_th)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = self.backbone(x)
        return self.cls_head(feat), self.reg_head(feat)


## 3. Deep Satellite Pansharpening Network (PanNet) & Metrics
Fuses lower-res Multispectral (MS) bands with crisp Panchromatic (PAN) band with SAM and ERGAS metrics.


In [ ]:
# --- 3. PanNet Residual Fusion & Remote Sensing Metrics ---
class CadastralPanSharpenNet(nn.Module):
    def __init__(self, ms_channels: int = 4, pan_channels: int = 1, num_filters: int = 32):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(ms_channels + pan_channels, num_filters, 3, padding=1),
            nn.BatchNorm2d(num_filters),
            nn.ReLU(inplace=True),
        )
        self.tail = nn.Conv2d(num_filters, ms_channels, 3, padding=1)

    def forward(self, ms: torch.Tensor, pan: torch.Tensor) -> torch.Tensor:
        if ms.shape[-2:] != pan.shape[-2:]:
            ms_up = F.interpolate(ms, size=pan.shape[-2:], mode="bilinear", align_corners=False)
        else:
            ms_up = ms

        pan_hp = pan - F.avg_pool2d(pan, 5, 1, 2)
        ms_hp = ms_up - F.avg_pool2d(ms_up, 5, 1, 2)

        x_in = torch.cat([ms_hp, pan_hp], dim=1)
        residual = self.tail(self.head(x_in))
        return torch.clamp(ms_up + residual, 0.0, 1.0)

def compute_spectral_angle_mapper(pred: torch.Tensor, target: torch.Tensor) -> float:
    dot = torch.sum(pred * target, dim=1)
    norm = torch.norm(pred, dim=1) * torch.norm(target, dim=1) + 1e-7
    sam = torch.acos(torch.clamp(dot / norm, -1.0, 1.0))
    return float(torch.mean(torch.rad2deg(sam)).item())

def compute_ergas(pred: torch.Tensor, target: torch.Tensor, ratio: float = 4.0) -> float:
    c = pred.shape[1]
    rmse_sum = 0.0
    for i in range(c):
        rmse = torch.sqrt(torch.mean((pred[:, i] - target[:, i]) ** 2))
        mean_t = torch.mean(target[:, i]) + 1e-7
        rmse_sum += (rmse / mean_t) ** 2
    return float((100.0 / ratio) * torch.sqrt((1.0 / c) * rmse_sum).item())


## 4. Atmospheric Cloud & Shadow Segmentation (CloudShadowNet)
Performs automated 4-class cloud, cirrus haze, and shadow quality assurance masking.


In [ ]:
# --- 4. CloudShadowNet ---
class CloudShadowNet(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


## 5. Geospatial Foundation Vision Transformer (GeoViT)
Extracts 256-D normalized semantic feature embeddings for Content-Based Image Retrieval (CBIR).


In [ ]:
# --- 5. GeoFoundation ViT & Remote Sensing Embedder ---
class GeoFoundationViT(nn.Module):
    def __init__(self, img_size: int = 128, patch_size: int = 16, embed_dim: int = 128, num_classes: int = 6):
        super().__init__()
        self.img_size = img_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True), num_layers=2
        )
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        b = x.shape[0]
        tokens = self.proj(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_tokens, tokens), dim=1) + self.pos_embed
        feats = self.encoder(x)[:, 0]
        return F.normalize(feats, p=2, dim=-1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        embs = self.forward_features(x)
        logits = self.classifier(embs)
        return logits, embs


## 🛰️ 6. Master Unified Pipeline: `UnifiedSatelliteFoundationEngine`
Consolidates all 5 satellite models into a single, comprehensive PyTorch remote sensing class.


In [ ]:
# --- Consolidated Master Model ---
class UnifiedSatelliteFoundationEngine(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 5):
        super().__init__()
        self.farseg = FarSeg(in_channels=in_channels, num_classes=num_classes)
        self.detector = OrientedCadastralDetector(in_channels=in_channels, num_classes=num_classes)
        self.pansharpen = CadastralPanSharpenNet(ms_channels=4, pan_channels=1)
        self.cloud_net = CloudShadowNet(in_channels=in_channels, num_classes=4)
        self.vit = GeoFoundationViT(img_size=128, embed_dim=128)
        self.eval()

    def run_full_pipeline(self, sat_rgb: np.ndarray) -> Dict[str, Any]:
        h, w = sat_rgb.shape[:2]
        img_norm = cv2.resize(sat_rgb, (128, 128)).astype(np.float32) / 255.0
        tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float()

        with torch.no_grad():
            farseg_logits = self.farseg(tensor)
            farseg_mask = torch.argmax(F.softmax(farseg_logits, dim=1), dim=1).squeeze(0).numpy()
            cloud_logits = self.cloud_net(tensor)
            cloud_mask = torch.argmax(F.softmax(cloud_logits, dim=1), dim=1).squeeze(0).numpy()
            logits, embs = self.vit(tensor)

        return {
            "farseg_land_cover_mask": farseg_mask,
            "cloud_shadow_mask": cloud_mask,
            "foundation_embedding": embs.squeeze(0).numpy(),
            "predicted_class_id": int(torch.argmax(logits, dim=-1).item()),
        }

# Instantiate and test
sat_master = UnifiedSatelliteFoundationEngine(num_classes=5)
dummy_sat_tile = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
results = sat_master.run_full_pipeline(dummy_sat_tile)

print("✅ Satellite Master Engine Pipeline Executed Successfully!")
print(f"FarSeg Land Cover Mask: {results['farseg_land_cover_mask'].shape}")
print(f"Cloud/Shadow Mask: {results['cloud_shadow_mask'].shape}")
print(f"Embedding Vector: {results['foundation_embedding'].shape} (L2 Normalized)")
